# Recording inventory: Mitt

Inspect dimensions, nonfinite values, continuous-channel distributions, and behavior event counts without loading a full session into memory. The paired raw files are required. The [raw event reader](../src/data_processing/vendor/events.py) defines the event and timestamp conventions.

Exploratory companion to the poster, not an input to the reviewed results. See the [README](../README.md#exploratory-notebooks) for the environment and data requirements.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / 'src/data_processing').is_dir()), None)
if repo is None:
    raise FileNotFoundError('Start Jupyter from the repository or its notebooks directory.')
sys.path.insert(0, str(repo / 'src'))
session = '080718_mitt'
raw = repo / 'data' / 'raw_data' / session
from data_processing.vendor.io import metadata, rows

lfp_path = raw / f'{session}_lfp.npz'
behavior_path = raw / f'{session}_bvr.npz'
lfp_meta, behavior_meta = (metadata(p) for p in (lfp_path, behavior_path))
for label, meta in [('LFP', lfp_meta), ('Behavior', behavior_meta)]:
    print(f'{label}: {meta["shape"][0]} rows × {meta["shape"][1]:,} samples, {meta["dtype"]}')

## Continuous channels and discrete events

Nonfinite values are counted separately and excluded from summaries. The X/Y channels are physical maze coordinates, not the Position target.

In [ ]:
def summarize(path, keys, continuous):
    summary, events = [], []
    for name, row in zip(keys, rows(path), strict=True):
        valid = row[np.isfinite(row)]
        info = {'Channel': name, 'Samples': row.size, 'Nonfinite': row.size - valid.size}
        if name in continuous:
            stats = dict.fromkeys(('Min', 'Q1', 'Median', 'Q3', 'Max', 'Mean', 'SD'), np.nan)
            if valid.size:
                quantiles = np.quantile(valid, [0, .25, .5, .75, 1])
                stats.update(zip(('Min', 'Q1', 'Median', 'Q3', 'Max'), quantiles, strict=True))
                stats.update(Mean=valid.mean(), SD=valid.std())
            summary.append(info | stats)
        else:
            values, counts = np.unique(valid, return_counts=True)
            events.append(info | {'Counts': dict(zip(values.tolist(), counts.tolist(), strict=True))})
    return pd.DataFrame(summary), pd.DataFrame(events)

continuous_behavior = {'TimeBin', 'XvalRatMazePosition', 'YvalRatMazePosition'}
lfp_summary, _ = summarize(lfp_path, lfp_meta['keys'], set(lfp_meta['keys']))
behavior_summary, event_counts = summarize(behavior_path, behavior_meta['keys'], continuous_behavior)
display(lfp_summary.round(4), behavior_summary.round(4), event_counts)

## LFP preview and recording gaps

The trace preview shows at most 1,000 samples per channel. Time gaps are reported from the recorded timestamps, not inferred from sequence labels.

In [ ]:
n_channels = len(lfp_meta['keys'])
fig, axes = plt.subplots(n_channels, 1, figsize=(11, .65 * n_channels), sharex=True, squeeze=False)
for ax, key, row in zip(axes[:, 0], lfp_meta['keys'], rows(lfp_path), strict=True):
    ax.plot(row[:1000], linewidth=.65)
    ax.set_ylabel(key.split('_')[0], rotation=0, labelpad=13)
axes[-1, 0].set_xlabel('Sample index')
fig.tight_layout()
plt.show()

from data_processing.vendor.events import time_segments

time = next(row for key, row in zip(behavior_meta['keys'], rows(behavior_path)) if key == 'TimeBin')
segments = time_segments(time)
print(f'{len(segments)} contiguous timestamp segments ({len(segments) - 1} gaps)')